# Probability Utils Functions

In [1]:
from math import log, ceil
import operator as op
from math import factorial as fac
from math import sqrt, log
import sys

class KEMParameterSet:
    def __init__(self, params: dict):
        # Basic parameters
        self.name  = params["name"]
        self.deg   = params["deg"]
        self.rank  = params["rank"]
        self.q     = params["q"]
        self.m     = params["m"]

        # Distributions
        self.s_param  = params["s_param"]
        self.e_param  = params["e_param"]
        self.r_param  = params["r_param"]
        self.e1_param = params["e1_param"]
        self.e2_param = params["e2_param"]

        # Compression params
        self.rpk_param = params["rpk_param"]
        self.rc1_param = params["rc1_param"]
        self.rc2_param = params["rc2_param"]

    def __repr__(self):
        cls_name = self.__class__.__name__
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls_name}({attrs})"

def summarize(ps):
    #print ("params: ", ps.__dict__)
    F, f = p2_cyclotomic_error_probability(ps)
    print(f"failure = 2^{log(f + 2.**(-1000))/log(2)}")
    return log(f + 2.**(-1000))/log(2)


def binomial(x, y):
    """ Binomial coefficient
    :param x: (integer)
    :param y: (integer)
    :returns: y choose x
    """
    try:
        binom = fac(x) // fac(y) // fac(x - y)
    except ValueError:
        binom = 0
    return binom


def centered_binomial_pdf(k, x):
    """ Probability density function of the centered binomial law of param k at x
    :param k: (integer)
    :param x: (integer)
    :returns: p_k(x)
    """
    return binomial(2*k, x+k) / 2.**(2*k)


  
def mod_switch(x, q, rq):
    """ Modulus switching (rounding to a different discretization of the Torus)
    :param x: value to round (integer)
    :param q: input modulus (integer)
    :param rq: output modulus (integer)
    """
    return int(round(1.* rq * x / q) % rq)


def mod_centered(x, q):
    """ reduction mod q, centered (ie represented in -q/2 .. q/2)
    :param x: value to round (integer)
    :param q: input modulus (integer)
    """
    a = x % q
    if a < q/2:
        return a
    return a - q


def build_mod_switching_error_law(q, rq):
    """ Construct Error law: law of the difference introduced by switching from and back a uniform value mod q
    :param q: original modulus (integer)
    :param rq: intermediate modulus (integer)
    """
    D = {}
    V = {}
    for x in range(q):
        y = mod_switch(x, q, rq)
        z = mod_switch(y, rq, q)
        d = mod_centered(x - z, q)
        D[d] = D.get(d, 0) + 1./q
        V[y] = V.get(y, 0) + 1

    return D


def law_convolution(A, B):
    """ Construct the convolution of two laws (sum of independent variables from two input laws)
    :param A: first input law (dictionnary)
    :param B: second input law (dictionnary)
    """

    C = {}
    for a in A:
        for b in B:
            c = a+b
            C[c] = C.get(c, 0) + A[a] * B[b]
    return C


def law_product(A, B):
    """ Construct the law of the product of independent variables from two input laws
    :param A: first input law (dictionnary)
    :param B: second input law (dictionnary)
    """
    C = {}
    for a in A:
        for b in B:
            c = a*b
            C[c] = C.get(c, 0) + A[a] * B[b]
    return C


def clean_dist(A):
    """ Clean a distribution to accelerate further computation (drop element of the support with proba less than 2^-300)
    :param A: input law (dictionnary)
    """
    B = {}
    for (x, y) in A.items():
        if y>2**(-300):
            B[x] = y
    return B


def iter_law_convolution(A, i):
    """ compute the -ith forld convolution of a distribution (using double-and-add)
    :param A: first input law (dictionnary)
    :param i: (integer)
    """
    D = {0: 1.0}
    i_bin = bin(i)[2:]  # binary representation of n
    for ch in i_bin:
        D = law_convolution(D, D)
        D = clean_dist(D)
        if ch == '1':
            D = law_convolution(D, A)
            D = clean_dist(D)
    return D


def tail_probability(D, t):
    '''
    Probability that an drawn from D is strictly greater than t in absolute value
    :param D: Law (Dictionnary)
    :param t: tail parameter (integer)
    '''
    s = 0
    ma = max(D.keys())
    if t >= ma:
        return 0
    for i in reversed(range(int(ceil(t)), ma)):  # Summing in reverse for better numerical precision (assuming tails are decreasing)
        s += D.get(i, 0) + D.get(-i, 0)
    return s


def build_centered_binomial_law(k):
    """ Construct the binomial law as a dictionnary
    :param k: (integer)
    :param x: (integer)
    :returns: A dictionnary {x:p_k(x) for x in {-k..k}}
    """
    D = {}
    for i in range(-k, k+1):
        D[i] = centered_binomial_pdf(k, i)
    return D


def build_HWT_law(hw):
    """ Construct the HWT law as a dictionnary
    :param hw: (float)
    :returns: A dictionnary {x:p_k(x) for x in {-1, 0, 1}}
    """
    D ={-1: hw/2, 0: 1-hw, 1: hw/2}
    return D


# Decryption Failure Probability Functions

In [ ]:


def p2_cyclotomic_final_error_distribution(ps):
    """ construct the final error distribution in our encryption scheme
    :param ps: parameter set (ParameterSet)
    """
    chis = build_centered_binomial_law(ps.s_param)           # Distribution law for secret vector s
    chir = chis.copy()                                       # Distribution law for the vector r in encryption
    chie = build_centered_binomial_law(ps.e_param)           # Distribution law for error vector e in KeyGen
    chie1 = chie.copy()                                      # Distribution law for error vector e1 in Enc
    chie2 = build_mod_switching_error_law(ps.q, ps.e2_param) # LWR distribution law used in the computation of 2nd ciphertext
    
    Rc1 = build_mod_switching_error_law(ps.q, ps.rc1_param)    # Compression error law used in the computation of 1st ciphertext

    # DFP expression of CCAKEM : e^T r + e2 - s^T e_1 - s^T c_u
    # e2 is the LWR error term and c_u is the compression error term (corresponding error law is Rc1 here)
    B1 = law_product(chie, chir)          # Distribution law for e^T r                 
    B2 = law_product(chis, chie1)         # Distribution law for s^T e_1
    B3 = law_product(chis, Rc1)           # Distribution law for s^T c_u

    # Distribution law for deg x rank copies
    C1 = iter_law_convolution(B1, ps.deg * ps.rank)        
    C2 = iter_law_convolution(B2, ps.deg * ps.rank)
    C3 = iter_law_convolution(B3, ps.deg * ps.rank)


    # Distribution law obtained from adding all the terms
    C=law_convolution(C1, C2)
    C = law_convolution(C, C3)

    D = law_convolution(C, chie2)                           # Final error
    return D

def p2_cyclotomic_error_probability(ps):
    F = p2_cyclotomic_final_error_distribution(ps)
    proba = tail_probability(F, ps.q/4)
    return F, ps.deg*proba

# Parameters and Corresponding DFP

In [3]:
ps_light = KEMParameterSet({
    "name": "CCAKEM-512",
    "deg": 256,
    "rank": 2,
    "q": 2048,
    "m": 512,
    "s_param": 1,
    "e_param": 3,
    "r_param": 1,
    "e1_param": 3,
    "e2_param": 2**4,
    "rpk_param": None,
    "rc1_param": 2**10,
    "rc2_param": None,
})

ps_recommended = KEMParameterSet({
    "name": "CCAKEM-768",
    "deg": 256,
    "rank": 3,
    "q": 4096,
    "m": 768,
    "s_param": 2,
    "e_param": 3,
    "r_param": 2,
    "e1_param": 3,
    "e2_param": 2**4,
    "rpk_param": None,
    "rc1_param": 2**10,
    "rc2_param": None,
})

ps_paranoid = KEMParameterSet({
    "name": "CCAKEM-1024",
    "deg": 256,
    "rank": 4,
    "q": 4096,
    "m": 1024,
    "s_param": 2,
    "e_param": 3,
    "r_param": 2,
    "e1_param": 3,
    "e2_param": 2**5,
    "rpk_param": None,
    "rc1_param": 2**11,
    "rc2_param": None,
})


# Analyses
print ("CCAKEM_512 :")
print ("--------------------")
summarize(ps_light)
print ()

print ("CCAKEM_768 :")
print ("--------------------")
summarize(ps_recommended)
print ()

print ("CCAKEM_1024 :")
print ("--------------------")
summarize(ps_paranoid)
print ()


CCAKEM_512 :
--------------------
failure = 2^-158.90129498454468

CCAKEM_768 :
--------------------
failure = 2^-165.65040829181018

CCAKEM_1024 :
--------------------
failure = 2^-181.78556887715132

